# Lezione 6 — Mappe interattive con Folium

## Perché questo ti riguarda?

Quando i tuoi dati hanno una posizione geografica — un indirizzo, delle coordinate, un paese — una tabella o un grafico a barre nasconde la parte più ovvia della storia: *dove* succedono le cose. Se hai 100 incidenti di polizia con latitudine e longitudine, o l'immigrazione in Canada suddivisa per paese, una mappa mostra pattern spaziali che nessun altro grafico può mostrare in modo così diretto.

## L'analogia

Folium è come pianificare un viaggio su una mappa fisica: prendi una mappa reale (non un disegno astratto di assi X e Y) e vai piantando spilli colorati nei luoghi che ti interessano. Ogni spillo può avere un'etichetta ("qui c'è stato un furto", "qui c'è un bar"). Se hai troppi spilli vicini, li raggruppi in un numero che dice "qui ci sono 40 punti" finché non ti avvicini abbastanza da vederli uno a uno. E se invece dei punti vuoi colorare intere regioni (paesi, stati) in base a un valore — più scuro dove il valore è più alto —, questa è una mappa **coropletica**: come colorare una mappa politica in base alla temperatura o alla popolazione di ogni paese.

Folium fa tutto questo su mappe reali e interattive (puoi fare zoom e spostarti), a differenza di Matplotlib, che disegna su una tela astratta.


## 0. Installazione e import delle librerie

In [ ]:
!pip install folium


In [ ]:
import numpy as np
import pandas as pd
import folium

print('Folium installato e importato!')


## 1. Creare una mappa base

Il primo passo è istanziare un oggetto `Map`. Senza argomenti, Folium ti dà la mappa mondiale completa.

In [ ]:
# mappa del mondo, senza centro definito
world_map = folium.Map()
world_map


Per centrare la mappa su un luogo specifico, le passi `location=[latitudine, longitudine]` e un livello di zoom con `zoom_start` (più è alto, più è ravvicinato).

In [ ]:
# mappa centrata sul Canada, zoom basso (vista ampia)
world_map = folium.Map(location=[56.130, -106.35], zoom_start=4)
world_map


In [ ]:
# lo stesso centro, ma con più zoom (vista più ravvicinata)
world_map = folium.Map(location=[56.130, -106.35], zoom_start=8)
world_map


Puoi anche cambiare lo stile visivo della mappa con il parametro `tiles`, ad esempio uno stile scuro ad alto contrasto (`'Cartodb dark_matter'`) o uno minimalista chiaro (`'Cartodb positron'`).

In [ ]:
# stile scuro, alto contrasto
world_map_dark = folium.Map(location=[56.130, -106.35], zoom_start=4,
                             tiles='Cartodb dark_matter')
world_map_dark


In [ ]:
# stile chiaro e minimalista
world_map_light = folium.Map(location=[56.130, -106.35], zoom_start=4,
                              tiles='Cartodb positron')
world_map_light


## 2. Aggiungere marcatori

Con la mappa base pronta, il passo successivo è "piantare gli spilli". Useremo il dataset degli incidenti di polizia di San Francisco del 2016 (`df_incidents`), che ha le colonne `Y` (latitudine), `X` (longitudine) e `Category` (tipo di incidente). Per costo computazionale, lavoriamo solo con i primi 100 record.

Nota: il portale originale di dati aperti di San Francisco (data.sfgov.org) non ospita più questo dataset con questa struttura dopo un aggiornamento dei sistemi del dipartimento di polizia. Tuttavia, la copia usata in questo corso (ospitata nel bucket di Skills Network / IBM) rimane disponibile pubblicamente, quindi la usiamo così com'è.

In [ ]:
df_incidents = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Police_Department_Incidents_-_Previous_Year__2016_.csv')

print('Dataset scaricato e letto in un dataframe di pandas!')
df_incidents.head()


In [ ]:
# tenere solo i primi 100 incidenti (per non saturare la mappa)
limit = 100
df_incidents = df_incidents.iloc[0:limit, :]
df_incidents.shape


In [ ]:
# coordinate di San Francisco
latitude = 37.77
longitude = -122.42

sanfran_map = folium.Map(location=[latitude, longitude], zoom_start=12)
sanfran_map


Il modo in cui Folium sovrappone più punti è creare un **feature group** (gruppo di elementi) e aggiungergli ogni marcatore.

In [ ]:
# gruppo di elementi per gli incidenti
incidents = folium.map.FeatureGroup()

for lat, lng in zip(df_incidents.Y, df_incidents.X):
    incidents.add_child(
        folium.vector_layers.CircleMarker(
            [lat, lng],
            radius=5,          # dimensione del cerchio
            color='yellow',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6
        )
    )

# aggiungere l'intero gruppo alla mappa
sanfran_map.add_child(incidents)


Affinché ogni marcatore mostri informazioni al passaggio del mouse (un **popup**), aggiungi marcatori di testo alle stesse coordinate.

In [ ]:
sanfran_map = folium.Map(location=[latitude, longitude], zoom_start=12)

# gruppo di elementi per gli incidenti (cerchi)
incidents = folium.map.FeatureGroup()
for lat, lng in zip(df_incidents.Y, df_incidents.X):
    incidents.add_child(
        folium.vector_layers.CircleMarker(
            [lat, lng],
            radius=5,
            color='yellow',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6
        )
    )
sanfran_map.add_child(incidents)

# marcatori con popup che mostrano la categoria dell'incidente
latitudes = list(df_incidents.Y)
longitudes = list(df_incidents.X)
labels = list(df_incidents.Category)

for lat, lng, label in zip(latitudes, longitudes, labels):
    folium.Marker([lat, lng], popup=label).add_to(sanfran_map)

sanfran_map


Un'alternativa più semplice, se la mappa appare troppo affollata con due livelli di marcatori, è mettere il `popup` direttamente nel `CircleMarker`.

In [ ]:
sanfran_map = folium.Map(location=[latitude, longitude], zoom_start=12)

for lat, lng, label in zip(df_incidents.Y, df_incidents.X, df_incidents.Category):
    folium.vector_layers.CircleMarker(
        [lat, lng],
        radius=5,
        color='yellow',
        fill=True,
        popup=label,
        fill_color='blue',
        fill_opacity=0.6
    ).add_to(sanfran_map)

sanfran_map


## 3. Raggruppare marcatori con MarkerCluster

Con molti punti, la mappa si satura visivamente. La soluzione di Folium è raggrupparli in cluster: quando sei lontano, vedi un numero (quanti incidenti ci sono in quella zona); quando ti avvicini con lo zoom, il cluster si apre gradualmente fino a mostrare i marcatori individuali.

In [ ]:
from folium import plugins

sanfran_map = folium.Map(location=[latitude, longitude], zoom_start=12)

# oggetto MarkerCluster aggiunto direttamente alla mappa
incidents = plugins.MarkerCluster().add_to(sanfran_map)

for lat, lng, label in zip(df_incidents.Y, df_incidents.X, df_incidents.Category):
    folium.Marker(
        location=[lat, lng],
        icon=None,
        popup=label,
    ).add_to(incidents)

sanfran_map


Allontanando completamente lo zoom, vedi un unico cluster con il totale (100, in questo caso). Avvicinandoti, quel cluster si scompone in cluster più piccoli fino ad arrivare ai marcatori individuali.

## 4. Choropleth: colorare regioni in base a un valore

Una mappa **coropletica** non colloca punti, colora regioni intere (paesi, stati, province) in base a un valore numerico: più scuro (o più rosso, a seconda della palette) dove il valore è più alto. Costruiamone una con il totale dell'immigrazione in Canada per paese, usando `df_can`.

In [ ]:
df_can = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv')

print('Dati scaricati e letti in un dataframe!')
df_can.head()


Per la mappa coropletica ci serve un file **GeoJSON** che definisca i confini geografici di ogni paese (in questo caso, `world_countries.json`).

In [ ]:
# GeoJSON con i confini di tutti i paesi del mondo
world_geo = r'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/world_countries.json'

# mappa base centrata su [0, 0] con zoom molto ampio (vista mondiale)
world_map = folium.Map(location=[0, 0], zoom_start=2)
world_map


`folium.Choropleth` ha bisogno di quattro elementi chiave: il GeoJSON (`geo_data`), il DataFrame con i dati (`data`), quali colonne usare (`columns`: la chiave che coincide con il GeoJSON e il valore da colorare), e `key_on`, che indica quale proprietà del GeoJSON contiene il nome che fa da collegamento (in questo file, `feature.properties.name`, e deve coincidere esattamente, maiuscole comprese, con i nomi dei paesi usati nel GeoJSON). Nelle versioni recenti di Folium, `Choropleth` è un livello che si crea a parte e si aggiunge alla mappa con `.add_to(mappa)` (il vecchio metodo `mappa.choropleth(...)` non esiste più in Folium 0.14+).

In [ ]:
folium.Choropleth(
    geo_data=world_geo,
    data=df_can,
    columns=['Country', 'Total'],       # colonna di collegamento e colonna di valore
    key_on='feature.properties.name',   # proprietà del GeoJSON che coincide con 'Country'
    fill_color='YlOrRd',                # palette di colori (giallo-arancione-rosso)
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Immigration to Canada'
).add_to(world_map)

world_map

Il risultato: più un paese è rosso/scuro, più immigrati ha inviato in Canada tra il 1980 e il 2013. Con questo dataset, i paesi più scuri sono Cina, India e Filippine, seguiti da Polonia, Pakistan e Stati Uniti.

## Punti chiave

- `folium.Map(location=[lat, lon], zoom_start=n)` crea la mappa base; senza `location` ottieni la mappa mondiale completa.
- Il parametro `tiles` cambia lo stile visivo della mappa (ad esempio, `'Cartodb dark_matter'` o `'Cartodb positron'`).
- `folium.Marker(...)` e `folium.vector_layers.CircleMarker(...)` aggiungono punti alla mappa; il parametro `popup` mostra testo quando interagisci con il marcatore.
- Un `folium.map.FeatureGroup()` ti permette di raggruppare più elementi e aggiungerli alla mappa in un colpo solo con `add_child`.
- `plugins.MarkerCluster()` raggruppa automaticamente marcatori vicini in un numero che si apre man mano che fai zoom, evitando mappe sature.
- Una mappa **coropletica** (`folium.Choropleth(...).add_to(mappa)`) colora regioni intere in base a un valore, usando un file GeoJSON come base dei confini e `key_on` per collegare i tuoi dati alle proprietà del GeoJSON.
- Il nome in `key_on` deve coincidere esattamente (maiuscole e formato) con quello usato nel GeoJSON, altrimenti la mappa non colorerà nulla.